# 06 — Survey: which tissues and cell types can be tested?

Before running the pipeline on new tissues, count what each tissue can support. Only the cell metadata (`obs`) is read, so this is fast. The rules below were fixed **before** any new tissue was analyzed, so a tissue can't be included or dropped after seeing its result.

| Level | Rule | Why |
| --- | --- | --- |
| classifiable | ≥ 2 young and ≥ 2 old mice, ≥ 50 cells per group | mouse-held-out folds need 2 mice per side |
| batch-testable | classifiable, plus ≥ 2 distinct young ages and ≥ 2 distinct old ages | the cross-age check needs an unseen age group on both sides |

Young = 1m, 3m. Old = 18m, 21m, 24m, 30m. Writes `figure/atlas/survey_*.csv`.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import anndata as ad
import numpy as np
import pandas as pd
from src import aging as ag

A = ag.paths("atlas")          # figure/atlas/ holds the cross-tissue tables

### 2. Read `obs` from every downloaded droplet tissue

`backed="r"` leaves the expression matrix on disk, so even Marrow (1.3 GB) loads in seconds.

In [ ]:
obs = {}
for d in sorted((ag.ROOT / "data").iterdir()):
    p = ag.paths(d.name).official
    if p.exists():
        a = ad.read_h5ad(p, backed="r")
        obs[d.name] = a.obs[["age", "mouse.id", "sex", "cell_ontology_class"]].copy()
        a.file.close()
print(len(obs), "tissues:", list(obs))

### 3. Mice per age, per tissue

In [ ]:
mice = pd.concat({t: o.drop_duplicates("mouse.id") for t, o in obs.items()}, names=["tissue"]).reset_index(level=0)
mice["age"] = mice["age"].astype(str)
per_age = pd.crosstab(mice["tissue"], mice["age"])[[a for a in ag.AGE_MONTHS if a in set(mice["age"])]]
per_age["young_mice"] = per_age[[a for a in ag.YOUNG if a in per_age]].sum(axis=1)
per_age["old_mice"] = per_age[[a for a in ag.OLD if a in per_age]].sum(axis=1)
per_age["cells"] = pd.Series({t: len(o) for t, o in obs.items()})
per_age["batch_testable_ages"] = pd.Series({t: len(ag.cross_age_splits(o["age"].astype(str).unique())) > 0
                                            for t, o in obs.items()})
per_age.to_csv(A.figure("survey_mice_per_age.csv"))
per_age.sort_values("cells", ascending=False)

### 4. Sex balance per age (counted in mice)

Where an age group is one sex only, age and sex are confounded in that tissue, as in Lung.

In [ ]:
sex = mice.groupby(["tissue", "age"], observed=True)["sex"].agg(lambda s: f"{(s == 'female').sum()}F {(s == 'male').sum()}M")
sex.unstack()

### 5. Testable cell types per tissue

`ag.usable_cell_types` only reads `obs`, so it runs on the backed metadata. `min_ages=1` gives the classifiable cell types; `min_ages=2` the batch-testable ones.

In [ ]:
class _Obs:                      # minimal stand-in: usable_cell_types only needs .obs
    def __init__(self, o): self.obs = o

rows = []
for t, o in obs.items():
    classifiable = set(ag.usable_cell_types(_Obs(o)))
    testable = set(ag.usable_cell_types(_Obs(o), min_ages=2))
    for ct in sorted(classifiable):
        rows.append({"tissue": t, "cell_type": ct, "batch_testable": ct in testable,
                     "n_cells": int((o["cell_ontology_class"] == ct).sum()),
                     "n_mice": o.loc[o["cell_ontology_class"] == ct, "mouse.id"].nunique()})
testable = pd.DataFrame(rows)
testable.to_csv(A.figure("survey_cell_types.csv"), index=False)
summary = testable.groupby("tissue").agg(classifiable=("cell_type", "size"), batch_testable=("batch_testable", "sum"))
summary.sort_values("batch_testable", ascending=False)

### 6. Tissues with nothing testable, and why

In [ ]:
none = per_age.loc[~per_age.index.isin(summary.index[summary["batch_testable"] > 0]),
                   ["young_mice", "old_mice", "batch_testable_ages"] + [a for a in ag.AGE_MONTHS if a in per_age]]
none

### 7. Do tissues share mice?

TMS took many organs from each mouse. Shared mice mean tissues are **not independent replicates**: a batch effect in one mouse reaches every tissue it gave. The atlas therefore counts mice, not tissues, when it asks whether a pattern is general.

In [ ]:
share = pd.crosstab(mice["mouse.id"], mice["tissue"])
print("mice:", share.shape[0], "| tissues per mouse:", share.sum(axis=1).describe()[["min", "50%", "max"]].to_dict())
share.sum(axis=1).sort_values(ascending=False).head(10)

### Findings

Written after the survey has run on all downloaded tissues; key numbers go in `README.md`.